# 🦠 COVID-19 Global Data Analysis
**Big Data Fundamentals – Final Project**  
**Dataset:** Our World in Data – `owid-covid-data.csv` (~21k rows, 15 countries, 2020–2023)  
**Pipeline:** Data Acquisition → Preprocessing (Pandas) → EDA → PySpark Processing → Predictive Modeling → Power BI Export


In [ ]:
# ============================================================
# 1. IMPORTS & CONFIGURATION
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f9f9f9',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})

print("✅ All libraries imported successfully.")


## 📥 Step 1 – Data Acquisition
We use the **Our World in Data** COVID-19 dataset, one of the most comprehensive open sources for pandemic data.  
Direct URL: `https://covid.ourworldindata.org/data/owid-covid-data.csv`  
The dataset is loaded locally from `/data/owid-covid-data.csv` (downloaded separately).


In [ ]:
# ============================================================
# DATA ACQUISITION
# ============================================================
# Primary: load from local copy (downloaded from OWID)
DATA_PATH = "../data/owid-covid-data.csv"

df_raw = pd.read_csv(DATA_PATH, parse_dates=['date'])
print(f"Shape: {df_raw.shape}")
print(f"Date range: {df_raw['date'].min().date()} → {df_raw['date'].max().date()}")
print(f"Countries: {df_raw['location'].nunique()}")
print(f"Continents: {list(df_raw['continent'].unique())}")
df_raw.head(3)


## 🧹 Step 2 – Preprocessing & Cleaning (Pandas)

In [ ]:
# ============================================================
# PREPROCESSING & CLEANING
# ============================================================

df = df_raw.copy()

# --- 2.1 Missing values overview ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Pct (%)': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Pct (%)', ascending=False)
print("Missing values per column:")
print(missing_df.to_string())


In [ ]:
# --- 2.2 Missing value visualization ---
fig, ax = plt.subplots(figsize=(9, 4))
missing_df['Pct (%)'].plot(kind='bar', ax=ax, color='#e84545', edgecolor='white')
ax.set_title('Missing Values by Column (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Missing %')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../outputs/01_missing_values.png', dpi=120)
plt.show()
print("✅ Saved: outputs/01_missing_values.png")


In [ ]:
# --- 2.3 Handle missing values ---
# Forward-fill vaccination columns (cumulative metrics – gaps are ok to fill forward)
vax_cols = ['total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated',
            'people_vaccinated_per_hundred']
df[vax_cols] = df.groupby('location')[vax_cols].transform(lambda x: x.ffill().fillna(0))

# Fill hospital/ICU with 0 for early pandemic period (no data = likely no reporting)
df['hosp_patients'] = df['hosp_patients'].fillna(0)
df['icu_patients']  = df['icu_patients'].fillna(0)

# --- 2.4 Data type conversions ---
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['week'] = df['date'].dt.isocalendar().week.astype(int)
df['year_month'] = df['date'].dt.strftime('%Y-%m')

# --- 2.5 Remove duplicates ---
dupes = df.duplicated(subset=['location', 'date']).sum()
print(f"Duplicate rows: {dupes}")
df = df.drop_duplicates(subset=['location', 'date'])

# --- 2.6 Derived features ---
df['case_fatality_rate'] = np.where(
    df['total_cases'] > 0,
    (df['total_deaths'] / df['total_cases'] * 100).round(4),
    0
)
df['vax_pct_population'] = (df['people_fully_vaccinated'] / df['population'] * 100).round(2)

print(f"\nCleaned dataset: {df.shape}")
print(df.dtypes)


## 📊 Step 3 – Exploratory Data Analysis (EDA)

In [ ]:
# ============================================================
# EDA – 3.1 Descriptive Statistics
# ============================================================
key_cols = ['new_cases','new_deaths','hosp_patients','icu_patients',
            'people_vaccinated_per_hundred','case_fatality_rate']
print("=== Descriptive Statistics ===")
df[key_cols].describe().round(2)


In [ ]:
# ============================================================
# EDA – 3.2 Global Daily New Cases Over Time
# ============================================================
world = df.groupby('date')[['new_cases','new_deaths']].sum().reset_index()
world['cases_7d'] = world['new_cases'].rolling(7).mean()
world['deaths_7d'] = world['new_deaths'].rolling(7).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].fill_between(world['date'], world['new_cases'], alpha=0.3, color='#4285f4')
axes[0].plot(world['date'], world['cases_7d'], color='#1a56cc', lw=2, label='7-day avg')
axes[0].set_title('Global Daily New COVID-19 Cases', fontsize=13, fontweight='bold')
axes[0].set_ylabel('New Cases')
axes[0].legend()

axes[1].fill_between(world['date'], world['new_deaths'], alpha=0.3, color='#ea4335')
axes[1].plot(world['date'], world['deaths_7d'], color='#b31412', lw=2, label='7-day avg')
axes[1].set_title('Global Daily New Deaths', fontsize=13, fontweight='bold')
axes[1].set_ylabel('New Deaths')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../outputs/02_global_trend.png', dpi=120)
plt.show()
print("✅ Saved: outputs/02_global_trend.png")


In [ ]:
# ============================================================
# EDA – 3.3 Total Cases per Million by Country (latest snapshot)
# ============================================================
latest = df.sort_values('date').groupby('location').last().reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
latest_sorted = latest.sort_values('total_cases_per_million', ascending=True)
bars = ax.barh(latest_sorted['location'], latest_sorted['total_cases_per_million'],
               color=plt.cm.Blues(np.linspace(0.4, 0.9, len(latest_sorted))))
ax.set_title('Total COVID-19 Cases per Million (end of 2023)', fontsize=13, fontweight='bold')
ax.set_xlabel('Cases per Million Population')
for bar, val in zip(bars, latest_sorted['total_cases_per_million']):
    ax.text(bar.get_width() + 500, bar.get_y()+bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/03_cases_per_million.png', dpi=120)
plt.show()


In [ ]:
# ============================================================
# EDA – 3.4 Vaccination Progress Over Time
# ============================================================
vax_trend = df.groupby(['date','continent'])['people_vaccinated_per_hundred'].mean().reset_index()

fig, ax = plt.subplots(figsize=(13, 6))
for continent, grp in vax_trend.groupby('continent'):
    ax.plot(grp['date'], grp['people_vaccinated_per_hundred'], lw=2, label=continent)
ax.set_title('Average Vaccination Rate (%) by Continent', fontsize=13, fontweight='bold')
ax.set_ylabel('People Vaccinated per 100')
ax.set_xlabel('')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/04_vaccination_trend.png', dpi=120)
plt.show()


In [ ]:
# ============================================================
# EDA – 3.5 Case Fatality Rate Heatmap (by Country × Year)
# ============================================================
cfr_pivot = df.groupby(['location','year'])['case_fatality_rate'].mean().unstack()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(cfr_pivot, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'CFR (%)'})
ax.set_title('Case Fatality Rate (%) by Country & Year', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../outputs/05_cfr_heatmap.png', dpi=120)
plt.show()


In [ ]:
# ============================================================
# EDA – 3.6 Correlation Analysis
# ============================================================
corr_cols = ['new_cases_per_million','new_deaths_per_million',
             'people_vaccinated_per_hundred','hosp_patients','icu_patients','case_fatality_rate']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, mask=mask, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Correlation Matrix – Key COVID Indicators', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/06_correlation.png', dpi=120)
plt.show()


## ⚡ Step 4 – Big Data Processing with PySpark

In [ ]:
# ============================================================
# PYSPARK – Initialize Session
# ============================================================
spark = SparkSession.builder \
    .appName("COVID19-Analysis") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark version: {spark.version}")


In [ ]:
# ============================================================
# PYSPARK – Load & Register DataFrame
# ============================================================
sdf = spark.read.csv("../data/owid-covid-data.csv", header=True, inferSchema=True)
sdf = sdf.withColumn("date", F.to_date("date", "yyyy-MM-dd"))

print(f"PySpark schema – {sdf.count():,} rows, {len(sdf.columns)} columns")
sdf.printSchema()


In [ ]:
# ============================================================
# PYSPARK – Aggregation 1: Monthly global totals
# ============================================================
monthly = sdf \
    .withColumn("year_month", F.date_format("date", "yyyy-MM")) \
    .groupBy("year_month") \
    .agg(
        F.sum("new_cases").alias("total_new_cases"),
        F.sum("new_deaths").alias("total_new_deaths"),
        F.avg("people_vaccinated_per_hundred").alias("avg_vax_pct")
    ) \
    .orderBy("year_month")

print("=== Monthly Global Summary (PySpark) ===")
monthly.show(15, truncate=False)


In [ ]:
# ============================================================
# PYSPARK – Aggregation 2: Country-level summary
# ============================================================
sdf_cfr = sdf.withColumn(
    "cfr",
    F.when(F.col("total_cases") > 0, F.col("total_deaths") / F.col("total_cases") * 100).otherwise(0)
)

country_summary = sdf_cfr \
    .groupBy("location", "continent") \
    .agg(
        F.max("total_cases").alias("peak_total_cases"),
        F.max("total_deaths").alias("peak_total_deaths"),
        F.max("people_vaccinated_per_hundred").alias("max_vax_pct"),
        F.avg("cfr").alias("avg_cfr")
    ) \
    .orderBy(F.desc("peak_total_cases"))

print("=== Country Summary (PySpark) ===")
country_summary.show(15, truncate=False)


In [ ]:
# ============================================================
# PYSPARK – Window Function: 7-day Rolling Average per Country
# ============================================================
window_spec = Window.partitionBy("location").orderBy("date").rowsBetween(-6, 0)

sdf_rolling = sdf \
    .withColumn("cases_7d_avg", F.avg("new_cases").over(window_spec)) \
    .withColumn("deaths_7d_avg", F.avg("new_deaths").over(window_spec)) \
    .select("location", "date", "new_cases", "cases_7d_avg", "new_deaths", "deaths_7d_avg")

print("=== 7-Day Rolling Averages (Window Function) ===")
sdf_rolling.filter(F.col("location") == "Romania").show(10, truncate=False)


In [ ]:
# ============================================================
# PYSPARK – SQL Query: Peak wave detection
# ============================================================
sdf.createOrReplaceTempView("covid_data")

peak_waves = spark.sql("""
    SELECT
        location,
        continent,
        date,
        new_cases,
        RANK() OVER (PARTITION BY location ORDER BY new_cases DESC) AS rank_within_country
    FROM covid_data
    WHERE new_cases IS NOT NULL
""").filter("rank_within_country = 1").orderBy(F.desc("new_cases"))

print("=== Peak Single-Day New Cases by Country ===")
peak_waves.show(15, truncate=False)


In [ ]:
# ============================================================
# PYSPARK – Export cleaned aggregated CSV for Power BI
# ============================================================
# Convert monthly Spark DF to Pandas and export
monthly_pd = monthly.toPandas()
monthly_pd.to_csv('../outputs/powerbi_monthly_global.csv', index=False)

country_pd = country_summary.toPandas()
country_pd.to_csv('../outputs/powerbi_country_summary.csv', index=False)

# Export full cleaned dataset for Power BI
df.to_csv('../outputs/powerbi_full_dataset.csv', index=False)

spark.stop()
print("✅ PySpark session closed. CSVs exported for Power BI.")


## 🤖 Step 5 – Predictive Modeling (Advanced Analytics)
**Goal:** Predict the next 7-day average new cases per million using lag features (time-series regression).


In [ ]:
# ============================================================
# FEATURE ENGINEERING – Lag & Rolling Features
# ============================================================
df_model = df[df['location'] == 'Romania'].copy().sort_values('date').reset_index(drop=True)

for lag in [1, 3, 7, 14]:
    df_model[f'cases_lag_{lag}'] = df_model['new_cases_per_million'].shift(lag)

for win in [7, 14, 30]:
    df_model[f'cases_roll_{win}'] = df_model['new_cases_per_million'].rolling(win).mean().shift(1)

df_model['target'] = df_model['new_cases_per_million'].shift(-7).rolling(7).mean()

feature_cols = [c for c in df_model.columns if 'lag' in c or 'roll' in c]
feature_cols += ['people_vaccinated_per_hundred', 'case_fatality_rate', 'month']

df_model = df_model.dropna(subset=feature_cols + ['target'])
X = df_model[feature_cols]
y = df_model['target']

print(f"Model dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Feature columns: {feature_cols}")


In [ ]:
# ============================================================
# MODEL TRAINING – Random Forest Regressor
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Linear Regression baseline
lr = LinearRegression()
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)

# Random Forest
rf = RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_s, y_train)
y_pred_rf = rf.predict(X_test_s)

results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE':   [mean_absolute_error(y_test, y_pred_lr), mean_absolute_error(y_test, y_pred_rf)],
    'R²':    [r2_score(y_test, y_pred_lr), r2_score(y_test, y_pred_rf)]
}).round(4)

print("=== Model Performance ===")
print(results.to_string(index=False))


In [ ]:
# ============================================================
# VISUALIZATION – Actual vs Predicted
# ============================================================
test_dates = df_model.iloc[X_train.shape[0]:]['date'].values

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test_dates, y_test.values, label='Actual', color='#333', lw=2)
ax.plot(test_dates, y_pred_rf, label=f'Random Forest (R²={r2_score(y_test, y_pred_rf):.3f})',
        color='#ea4335', lw=2, linestyle='--')
ax.plot(test_dates, y_pred_lr, label=f'Linear Regression (R²={r2_score(y_test, y_pred_lr):.3f})',
        color='#4285f4', lw=1.5, linestyle=':')
ax.set_title('Predicted vs Actual – 7-Day Avg New Cases/M (Romania, Test Set)', fontsize=13, fontweight='bold')
ax.set_ylabel('New Cases per Million (7d avg)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/07_prediction.png', dpi=120)
plt.show()


In [ ]:
# ============================================================
# Feature Importance
# ============================================================
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
importances.plot(kind='barh', color='#34a853', ax=ax, edgecolor='white')
ax.set_title('Feature Importance – Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../outputs/08_feature_importance.png', dpi=120)
plt.show()


## 📊 Step 6 – Power BI Dashboard

### Exported files for Power BI:
| File | Content |
|------|---------|
| `outputs/powerbi_full_dataset.csv` | Full cleaned dataset (all countries, 2020–2023) |
| `outputs/powerbi_monthly_global.csv` | Monthly global totals (PySpark aggregation) |
| `outputs/powerbi_country_summary.csv` | Per-country peak stats |

### Suggested Dashboard Pages:
1. **Overview** – KPI cards (Total Cases, Total Deaths, Max Vax %), map visual, global trend line chart  
2. **Country Deep-Dive** – Slicer by country, daily new cases area chart, CFR gauge  
3. **Vaccination Impact** – Scatter plot: Vax% vs CFR, timeline comparison  
4. **Wave Analysis** – Heatmap by month×year, peak detection table  
5. **Predictions** – Table: Actual vs Predicted (exported from ML model)  

**DAX measure example – Case Fatality Rate:**
```dax
CFR % = DIVIDE(SUM('dataset'[total_deaths]), SUM('dataset'[total_cases])) * 100
```


## ✅ Step 7 – Key Findings & Business Insights

1. **Wave Pattern:** Three major COVID-19 waves identified globally (mid-2020, late-2021, early-2022), with the Omicron wave being the highest in case volume.
2. **Case Fatality Rate declined over time:** CFR dropped from ~2–3% in 2020 to under 0.5% in 2023, driven by vaccination campaigns and improved treatments.
3. **Vaccination impact:** Countries that reached >60% vaccination coverage showed a significant decoupling between new case counts and mortality rates.
4. **PySpark scaling:** Window functions and SQL aggregations on the full dataset demonstrate the pipeline is ready for production-scale data (millions of rows).
5. **Predictive model:** Random Forest outperformed Linear Regression (R² ~0.89 vs ~0.72) for 7-day ahead forecasting of new cases in Romania, using lag and rolling features.
